In [275]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [276]:
df=pd.read_csv(r"C:\Data Science Learning\Projects\regression-for-menstrual-pain\dataset\final_ds.csv")
df.head()

,cycle_phase_Luteal,cycle_phase_Menstrual,pms_symptoms_Yes,ovulation_result_Positive,user_id,cycle_number,start_date,cycle_length_days,prev_cycle_length,flow_level,...,diet_quality,exercise_frequency,sleep_hours,caffeine_intake,water_intake_liters,alcohol_consumption,smoking_status,birth_control_use,pcos_diagnosed,stress_score_baseline
0,0.0,0.0,0.0,0.0,U00001,2,3/17/2024,33,33.0,0.0,...,2.0,2.0,5.4,1.5,2.0,1.0,0,1,1,4.1
1,0.0,0.0,0.0,0.0,U00001,3,4/19/2024,34,33.0,2.0,...,2.0,2.0,5.4,1.5,2.0,1.0,0,1,1,4.1
2,1.0,0.0,0.0,0.0,U00001,4,5/23/2024,31,34.0,0.0,...,2.0,2.0,5.4,1.5,2.0,1.0,0,1,1,4.1
3,1.0,0.0,0.0,0.0,U00001,5,6/23/2024,31,31.0,1.0,...,2.0,2.0,5.4,1.5,2.0,1.0,0,1,1,4.1
4,1.0,0.0,0.0,0.0,U00001,6,7/24/2024,37,31.0,1.0,...,2.0,2.0,5.4,1.5,2.0,1.0,0,1,1,4.1


In [277]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [278]:
df.columns

Index(['cycle_phase_Luteal', 'cycle_phase_Menstrual', 'pms_symptoms_Yes',
       'ovulation_result_Positive', 'user_id', 'cycle_number', 'start_date',
       'cycle_length_days', 'prev_cycle_length', 'flow_level', 'pain_level',
       'mood_score', 'stress_score_cycle', 'sleep_hours_cycle', 'energy_level',
       'concentration_score', 'work_hours_lost', 'estrogen_pgml',
       'progesterone_ngml', 'overall_health_score', 'log_consistency_score',
       'prepared_before_period', 'state', 'age', 'bmi', 'diet_quality',
       'exercise_frequency', 'sleep_hours', 'caffeine_intake',
       'water_intake_liters', 'alcohol_consumption', 'smoking_status',
       'birth_control_use', 'pcos_diagnosed', 'stress_score_baseline'],
      dtype='object')

In [279]:
from sklearn.linear_model import LinearRegression,Lasso,Ridge,SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split,KFold,cross_val_score

In [280]:
X=df.drop(["pain_level","start_date","user_id","state"],axis=1)
X.columns

Index(['cycle_phase_Luteal', 'cycle_phase_Menstrual', 'pms_symptoms_Yes',
       'ovulation_result_Positive', 'cycle_number', 'cycle_length_days',
       'prev_cycle_length', 'flow_level', 'mood_score', 'stress_score_cycle',
       'sleep_hours_cycle', 'energy_level', 'concentration_score',
       'work_hours_lost', 'estrogen_pgml', 'progesterone_ngml',
       'overall_health_score', 'log_consistency_score',
       'prepared_before_period', 'age', 'bmi', 'diet_quality',
       'exercise_frequency', 'sleep_hours', 'caffeine_intake',
       'water_intake_liters', 'alcohol_consumption', 'smoking_status',
       'birth_control_use', 'pcos_diagnosed', 'stress_score_baseline'],
      dtype='object')

In [281]:
y=df["pain_level"]

In [282]:
y.head()

0    2
1    8
2    3
3    3
4    6
Name: pain_level, dtype: int64

In [283]:
X.drop(["cycle_number","prev_cycle_length"],axis=1,inplace=True)

In [284]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

In [285]:
pipeline=Pipeline([
    ("scaler",StandardScaler()),
    ("model",LinearRegression())
])

In [286]:
X_train.shape

(9012, 29)

In [287]:
kf=KFold(n_splits=10,shuffle=True,random_state=42)

In [288]:
cv_scores=cross_val_score(
    pipeline,
    X_train,y_train,
    cv=kf,
    scoring="r2"
)

In [289]:
pipeline.fit(X_train, y_train)          # train on full train+val
test_accuracy = pipeline.score(X_test, y_test)

print("\n=== Final Test Set Evaluation ===")
print(f"Test Accuracy : {test_accuracy:.4f}")


=== Final Test Set Evaluation ===
Test Accuracy : 0.9361


In [290]:
print("=== K-Fold Cross Validation Results ===")
print(f"Fold Scores : {np.round(cv_scores, 4)}")
print(f"Mean Accuracy : {cv_scores.mean():.4f}")
print(f"Std Dev       : {cv_scores.std():.4f}")


=== K-Fold Cross Validation Results ===
Fold Scores : [0.9362 0.9337 0.9379 0.9295 0.9391 0.9376 0.9375 0.9349 0.9414 0.9409]
Mean Accuracy : 0.9369
Std Dev       : 0.0034


In [291]:
def models(model,X_train,y_train,X_test,y_test,name,n):
    kf=KFold(n_splits=n,shuffle=True,random_state=42)
    
    pipeline=Pipeline([
    ("scaler",StandardScaler()),
    ("model",model)])

    cv_scores=cross_val_score(
    pipeline,
    X_train,y_train,
    cv=kf,
    scoring="r2")
    
    pipeline.fit(X_train,y_train)
    test_r2 = pipeline.score(X_test, y_test)

    print(f"\n=== Final Test Set Evaluation for {name} R2 Score===")
    print(f"Test Accuracy : {test_r2:.4f}")
    
    print(f"=== K-Fold Cross Validation Results for {name} R2 Score===")
    print(f"Fold Scores : {np.round(cv_scores, 4)}")
    print(f"Mean Accuracy : {cv_scores.mean():.4f}")
    print(f"Std Dev       : {cv_scores.std():.4f}")

    n_samples = X_test.shape[0]
    n_features = X_test.shape[1]
    
    adj_r2 = 1 - (1 - test_r2) * (n_samples - 1) / (n_samples - n_features - 1)
    print(f"Adjusted R2 Score for {name}: {adj_r2:.4f}")

In [292]:
Sgd=SGDRegressor(loss="squared_error",penalty=None,max_iter=500)
models(Sgd,X_train,y_train,X_test,y_test,"SGD",10)


=== Final Test Set Evaluation for SGD R2 Score===
Test Accuracy : 0.9357
=== K-Fold Cross Validation Results for SGD R2 Score===
Fold Scores : [0.936  0.9337 0.9369 0.9296 0.939  0.9372 0.9369 0.9347 0.9395 0.941 ]
Mean Accuracy : 0.9365
Std Dev       : 0.0031
Adjusted R2 Score for SGD: 0.9352


In [293]:
ridge=Ridge(alpha=0.95,solver="svd")
models(ridge,X_train,y_train,X_test,y_test,"Ridge",10)


=== Final Test Set Evaluation for Ridge R2 Score===
Test Accuracy : 0.9361
=== K-Fold Cross Validation Results for Ridge R2 Score===
Fold Scores : [0.9362 0.9337 0.9379 0.9295 0.9391 0.9376 0.9375 0.9349 0.9414 0.941 ]
Mean Accuracy : 0.9369
Std Dev       : 0.0034
Adjusted R2 Score for Ridge: 0.9356


In [294]:
lasso=Lasso(alpha=0.55,max_iter=50)
models(lasso,X_train,y_train,X_test,y_test,"Lasso",5)


=== Final Test Set Evaluation for Lasso R2 Score===
Test Accuracy : 0.8102
=== K-Fold Cross Validation Results for Lasso R2 Score===
Fold Scores : [0.8102 0.813  0.8119 0.8059 0.8135]
Mean Accuracy : 0.8109
Std Dev       : 0.0027
Adjusted R2 Score for Lasso: 0.8088


In [295]:
from sklearn.metrics import r2_score

In [296]:
def tune_lasso(X_train, y_train, X_test, y_test, 
               alphas=None, n_splits=10):
    
    if alphas is None:
        alphas = [0.0001, 0.001, 0.01, 0.1, 0.2, 0.3, 0.5, 1.0, 5.0, 10.0]
    
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    n_train, p = X_train.shape
    n_test     = X_test.shape[0]
    
    results = []
    
    for alpha in alphas:
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('model', Lasso(alpha=alpha, max_iter=100))
        ])
        
        cv_scores = cross_val_score(pipeline, X_train, y_train, 
                                    cv=kf, scoring='r2')
        
        pipeline.fit(X_train, y_train)
        
        # ── Train metrics ─────────────────────────────────────
        y_pred_train   = pipeline.predict(X_train)
        train_r2       = r2_score(y_train, y_pred_train)
        train_adj_r2   = 1 - (1 - train_r2) * (n_train - 1) / (n_train - p - 1)
        
        # ── Test metrics ──────────────────────────────────────
        y_pred_test    = pipeline.predict(X_test)
        test_r2        = r2_score(y_test, y_pred_test)
        test_adj_r2    = 1 - (1 - test_r2) * (n_test - 1) / (n_test - p - 1)
        
        results.append({
            'alpha'       : alpha,
            'mean_cv_r2'  : cv_scores.mean(),
            'std_cv_r2'   : cv_scores.std(),
            'train_r2'    : train_r2,
            'train_adj_r2': train_adj_r2,
            'test_r2'     : test_r2,
            'test_adj_r2' : test_adj_r2
        })
    
    best = max(results, key=lambda x: x['mean_cv_r2'])
    
    # ── Print all results ─────────────────────────────────────
    print("=" * 100)
    print(f"{'Alpha':<10} {'CV R²':<10} {'CV Std':<10} {'Train R²':<12} {'Train AdjR²':<14} {'Test R²':<10} {'Test AdjR²':<12}")
    print("=" * 100)
    for r in results:
        marker = " ← best" if r['alpha'] == best['alpha'] else ""
        print(f"{r['alpha']:<10} "
              f"{r['mean_cv_r2']:<10.4f} "
              f"{r['std_cv_r2']:<10.4f} "
              f"{r['train_r2']:<12.4f} "
              f"{r['train_adj_r2']:<14.4f} "
              f"{r['test_r2']:<10.4f} "
              f"{r['test_adj_r2']:<12.4f}{marker}")
    
    print("=" * 100)
    print(f"\n Best Alpha        : {best['alpha']}")
    print(f" CV R²             : {best['mean_cv_r2']:.4f}")
    print(f" Train R²          : {best['train_r2']:.4f}")
    print(f" Train Adjusted R² : {best['train_adj_r2']:.4f}")
    print(f" Test R²           : {best['test_r2']:.4f}")
    print(f" Test Adjusted R²  : {best['test_adj_r2']:.4f}")
    
    # ── Refit best pipeline ───────────────────────────────────
    best_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', Lasso(alpha=best['alpha'], max_iter=50))
    ])
    best_pipeline.fit(X_train, y_train)

    return best_pipeline, best['alpha']

best_model, best_alpha = tune_lasso(X_train, y_train, X_test, y_test)

best_model.named_steps["model"].coef_

Alpha      CV R²      CV Std     Train R²     Train AdjR²    Test R²    Test AdjR²  
0.0001     0.9369     0.0034     0.9374       0.9372         0.9361     0.9356      
0.001      0.9369     0.0033     0.9374       0.9372         0.9361     0.9356       ← best
0.01       0.9365     0.0034     0.9368       0.9366         0.9358     0.9353      
0.1        0.9163     0.0047     0.9165       0.9162         0.9167     0.9161      
0.2        0.8971     0.0048     0.8974       0.8970         0.8979     0.8971      
0.3        0.8750     0.0049     0.8753       0.8749         0.8756     0.8746      
0.5        0.8254     0.0052     0.8256       0.8251         0.8252     0.8239      
1.0        0.6490     0.0055     0.6494       0.6482         0.6476     0.6449      
5.0        -0.0010    0.0011     0.0000       -0.0032        -0.0004    -0.0080     
10.0       -0.0010    0.0011     0.0000       -0.0032        -0.0004    -0.0080     

 Best Alpha        : 0.001
 CV R²             : 0.9369
 T

array([-2.59183537e-03,  3.18238059e-03, -6.31142934e-03,  7.31294115e-03,
       -1.07508417e-03,  2.85875339e-01, -1.09977987e-01, -5.21280635e-01,
        1.67197163e-01, -1.45215024e-01, -1.71543539e-01,  4.95179456e-01,
       -1.78776704e-03,  0.00000000e+00, -1.69846662e+00,  0.00000000e+00,
        9.47387031e-03,  2.44465994e-03, -1.19409072e-01, -4.33550823e-03,
       -1.12517034e-02,  2.23960217e-01,  9.34196178e-04, -0.00000000e+00,
       -0.00000000e+00,  3.07287470e-03, -3.54774260e-03, -7.58195441e-02,
        4.14329330e-03])

In [342]:
coefs = best_model.named_steps['model'].coef_

coef_df = pd.DataFrame({
    'Feature'    : X_train.columns,
    'Coefficient': coefs
})
coef_df

,Feature,Coefficient
0,cycle_phase_Luteal,-0.002592
1,cycle_phase_Menstrual,0.003182
2,pms_symptoms_Yes,-0.006311
3,ovulation_result_Positive,0.007313
4,cycle_length_days,-0.001075
5,flow_level,0.285875
6,mood_score,-0.109978
7,stress_score_cycle,-0.521281
8,sleep_hours_cycle,0.167197
9,energy_level,-0.145215


In [297]:
# 1. Check if scaler was fit on full data before splitting
#    (common mistake)
# Your pipeline fits scaler inside cv — should be fine ✅

# 2. Check for duplicate rows between train and test
import pandas as pd

X_train_df = pd.DataFrame(X_train)
X_test_df  = pd.DataFrame(X_test)

duplicates = pd.merge(X_train_df, X_test_df, how='inner')
print("Duplicate rows between train & test:", len(duplicates))

# 3. Check if split was done BEFORE or AFTER any preprocessing
# Your split should be the FIRST thing you do on raw data

Duplicate rows between train & test: 0


In [298]:
df.shape

(12875, 35)

In [299]:
#Creating Partial Models to Compare Scores
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [300]:
# MODEL 1: Only biological variables

model1 = smf.ols(
    formula="""
    pain_level ~ cycle_length_days+cycle_phase_Luteal+cycle_phase_Menstrual+flow_level
    +pms_symptoms_Yes+estrogen_pgml+progesterone_ngml
    +ovulation_result_Positive
    """,
    data=df
).fit()


print("\nMODEL 1 SUMMARY (Biological):")
print(model1.summary())


MODEL 1 SUMMARY (Biological):
                            OLS Regression Results                            
Dep. Variable:             pain_level   R-squared:                       0.377
Model:                            OLS   Adj. R-squared:                  0.376
Method:                 Least Squares   F-statistic:                     972.2
Date:                Sun, 12 Apr 2026   Prob (F-statistic):               0.00
Time:                        15:56:09   Log-Likelihood:                -25512.
No. Observations:               12875   AIC:                         5.104e+04
Df Residuals:                   12866   BIC:                         5.111e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------

In [301]:
# MODEL 2: Add psychological factors

model2 = smf.ols(
    formula="""
    pain_level ~ 
    cycle_length_days+cycle_phase_Luteal+cycle_phase_Menstrual+flow_level
    +pms_symptoms_Yes+estrogen_pgml+progesterone_ngml
    +ovulation_result_Positive+mood_score+stress_score_cycle+stress_score_baseline
    +energy_level+concentration_score+overall_health_score+prepared_before_period
    
    """,
    data=df
).fit()

print("\nMODEL 2 SUMMARY (Biological + Psychological):")
print(model2.summary())


MODEL 2 SUMMARY (Biological + Psychological):
                            OLS Regression Results                            
Dep. Variable:             pain_level   R-squared:                       0.912
Model:                            OLS   Adj. R-squared:                  0.911
Method:                 Least Squares   F-statistic:                     8830.
Date:                Sun, 12 Apr 2026   Prob (F-statistic):               0.00
Time:                        15:56:09   Log-Likelihood:                -12946.
No. Observations:               12875   AIC:                         2.592e+04
Df Residuals:                   12859   BIC:                         2.604e+04
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------

In [302]:
# MODEL 3: Add lifestyle factors

model3 = smf.ols(
    formula="""
    pain_level ~ cycle_length_days+cycle_phase_Luteal+cycle_phase_Menstrual+flow_level
    +pms_symptoms_Yes+estrogen_pgml+progesterone_ngml
    +ovulation_result_Positive+mood_score+stress_score_cycle+stress_score_baseline
    +energy_level+concentration_score+overall_health_score+prepared_before_period
    +sleep_hours+diet_quality+exercise_frequency+sleep_hours_cycle+caffeine_intake+
    +alcohol_consumption+smoking_status+work_hours_lost+log_consistency_score
    
    """,
    data=df
).fit()

print("\nMODEL 3 SUMMARY (Add Lifestyle):")
print(model3.summary())


MODEL 3 SUMMARY (Add Lifestyle):
                            OLS Regression Results                            
Dep. Variable:             pain_level   R-squared:                       0.934
Model:                            OLS   Adj. R-squared:                  0.933
Method:                 Least Squares   F-statistic:                     7521.
Date:                Sun, 12 Apr 2026   Prob (F-statistic):               0.00
Time:                        15:56:09   Log-Likelihood:                -11103.
No. Observations:               12875   AIC:                         2.226e+04
Df Residuals:                   12850   BIC:                         2.244e+04
Df Model:                          24                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------

In [303]:
# Model4: - Full Model

model1 = smf.ols(
    formula="""
    pain_level ~ cycle_phase_Luteal+cycle_phase_Menstrual+pms_symptoms_Yes
    +ovulation_result_Positive+flow_level+mood_score+stress_score_cycle+sleep_hours_cycle
    +energy_level+concentration_score+estrogen_pgml+progesterone_ngml+overall_health_score
    +log_consistency_score+prepared_before_period+age+bmi+diet_quality+exercise_frequency+sleep_hours
    +caffeine_intake+water_intake_liters+alcohol_consumption+smoking_status+birth_control_use
    +pcos_diagnosed+stress_score_baseline+cycle_number
    """,
    data=df
).fit()


print("\n Full Model")
print(model1.summary())


 Full Model
                            OLS Regression Results                            
Dep. Variable:             pain_level   R-squared:                       0.928
Model:                            OLS   Adj. R-squared:                  0.928
Method:                 Least Squares   F-statistic:                     5929.
Date:                Sun, 12 Apr 2026   Prob (F-statistic):               0.00
Time:                        15:56:10   Log-Likelihood:                -11602.
No. Observations:               12875   AIC:                         2.326e+04
Df Residuals:                   12846   BIC:                         2.348e+04
Df Model:                          28                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept

In [304]:
df.columns

Index(['cycle_phase_Luteal', 'cycle_phase_Menstrual', 'pms_symptoms_Yes',
       'ovulation_result_Positive', 'user_id', 'cycle_number', 'start_date',
       'cycle_length_days', 'prev_cycle_length', 'flow_level', 'pain_level',
       'mood_score', 'stress_score_cycle', 'sleep_hours_cycle', 'energy_level',
       'concentration_score', 'work_hours_lost', 'estrogen_pgml',
       'progesterone_ngml', 'overall_health_score', 'log_consistency_score',
       'prepared_before_period', 'state', 'age', 'bmi', 'diet_quality',
       'exercise_frequency', 'sleep_hours', 'caffeine_intake',
       'water_intake_liters', 'alcohol_consumption', 'smoking_status',
       'birth_control_use', 'pcos_diagnosed', 'stress_score_baseline'],
      dtype='object')

In [305]:
df.head()

,cycle_phase_Luteal,cycle_phase_Menstrual,pms_symptoms_Yes,ovulation_result_Positive,user_id,cycle_number,start_date,cycle_length_days,prev_cycle_length,flow_level,...,diet_quality,exercise_frequency,sleep_hours,caffeine_intake,water_intake_liters,alcohol_consumption,smoking_status,birth_control_use,pcos_diagnosed,stress_score_baseline
0,0.0,0.0,0.0,0.0,U00001,2,3/17/2024,33,33.0,0.0,...,2.0,2.0,5.4,1.5,2.0,1.0,0,1,1,4.1
1,0.0,0.0,0.0,0.0,U00001,3,4/19/2024,34,33.0,2.0,...,2.0,2.0,5.4,1.5,2.0,1.0,0,1,1,4.1
2,1.0,0.0,0.0,0.0,U00001,4,5/23/2024,31,34.0,0.0,...,2.0,2.0,5.4,1.5,2.0,1.0,0,1,1,4.1
3,1.0,0.0,0.0,0.0,U00001,5,6/23/2024,31,31.0,1.0,...,2.0,2.0,5.4,1.5,2.0,1.0,0,1,1,4.1
4,1.0,0.0,0.0,0.0,U00001,6,7/24/2024,37,31.0,1.0,...,2.0,2.0,5.4,1.5,2.0,1.0,0,1,1,4.1
